# Exploring weather delays with SQL

Investigate how often weather delays are reported, how long they last, and how they vary across airports and months.

### SQL building blocks

| Clause or expression | Purpose |
| --- | --- |
| `SELECT column AS name` | Choose a column or calculation and name the result |
| `FROM flights` | Choose the table |
| `WHERE condition` | Keep rows matching a condition |
| `COUNT(*)` | Count all rows, including rows with missing values |
| `COUNT(column)` | Count rows where that column is not `NULL` |
| `SUM`, `AVG`, `MIN`, `MAX` | Summarise values; these ignore `NULL` |
| `CASE WHEN ... THEN ... ELSE ... END` | Express an if/else calculation |
| `GROUP BY column` | Calculate a summary for each group |
| `HAVING condition` | Filter groups after aggregation |
| `ORDER BY value DESC` | Sort from largest to smallest |
| `LIMIT 10` | Return only ten rows |

Write clauses in this order: `SELECT`, `FROM`, `WHERE`, `GROUP BY`, `HAVING`, `ORDER BY`, `LIMIT`. A query conceptually filters rows before forming groups. SQL `NULL` means missing or unknown: use `IS NULL`, never `= NULL`.

## 1. Load the data

One row represents a scheduled flight record. We will use:

| Column | Meaning |
| --- | --- |
| `flight_date_local_origin` | Flight date at the origin |
| `origin_iata`, `dest_iata` | Departure and destination airport codes |
| `dep_delay_min` | Departure delay in minutes; negative values mean early departures |
| `reported_weather_delay_min` | Minutes attributed to weather in the delay-cause reporting |
| `cancelled`, `cancellation_code` | Cancellation flag and reason code (`B` means weather) |
| `wx_origin_obs_missing` | Whether the origin weather observation is missing |
| `wx_origin_rain`, `wx_origin_snow`, `wx_origin_thunderstorm` | Observed weather flags at the origin |

In [ ]:
from pathlib import Path
import calendar
import duckdb
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Start in the project or notebooks directory.'

YEAR = 2023
TOP_N_AIRPORTS = 10  # maximum airports shown in each ranking

files = sorted((ROOT / 'data' / 'bts_flights_weather' / 'flights_weather' / f'year={YEAR}').glob('*.parquet'))
assert files, 'Download the BTS flights/weather data first (see README.md).'

con = duckdb.connect()  # An in-memory database; source files remain unchanged.
con.execute("SET threads = 4")
con.execute("SET memory_limit = '2GB'")
# Register the files under a table-like name usable in SQL.
con.read_parquet([str(file) for file in files], hive_partitioning=False).create_view('flights')

# Configure matplotlib
plt.rcParams['figure.figsize'] = (10, 4) # configure the size fo the figures

### List the available tables

Use a SQL command to list the tables registered in the database. Which table contains the flight records?

**Result table:** `tables_name`  
**Column names (in order):** `name`.

In [ ]:
# Write the SQL query below. Keep these variable names and column aliases for the charts.
sql_tables_name = """
    SHOW TABLES;
"""
if not sql_tables_name.strip():
    raise NotImplementedError("Add a SQL query to continue: List the available tables.")
tables_name = con.execute(sql_tables_name).fetchdf()
display(tables_name)

### Inspect the table structure

Inspect the column names and data types in `flights`. Identify the date column, the two airport-code columns, and the weather-delay column.

**Result table:** `columns_name`  
**Column names (in order):** `column_name`, `column_type`, `null`, `key`, `default`, `extra`.

In [ ]:
# Write the SQL query below. Keep these variable names and column aliases for the charts.
sql_columns_name = """
DESCRIBE flights;
"""
if not sql_columns_name.strip():
    raise NotImplementedError("Add a SQL query to continue: Inspect the table structure.")
columns_name = con.execute(sql_columns_name).fetchdf()
display(columns_name)
print(columns_name['column_name'].tolist()[0:10])

### Identify the weather variables at both airports

In [ ]:
sql_weather_preview = """

"""
if not sql_weather_preview.strip():
    raise NotImplementedError("Add a SQL query to inspect origin and destination weather.")
weather_preview = con.execute(sql_weather_preview).fetchdf()
display(weather_preview.head(10))

### Look at five flight records

Select the flight date, origin, destination, departure delay, and reported weather-delay minutes. Return only five rows. Do not change the source data.

**Result table:** `preview`  
**Column names (in order):** `flight_date_local_origin`, `origin_iata`, `dest_iata`, `dep_delay_min`, `reported_weather_delay_min`.

In [ ]:
# Write the SQL query below. Keep these variable names and column aliases for the charts.
sql_preview = """

"""
if not sql_preview.strip():
    raise NotImplementedError("Add a SQL query to continue: Look at five flight records.")
preview = con.execute(sql_preview).fetchdf()
display(preview)

## 2. Are values missing ?

### Check missing and unexpected values

Return one row with the total number of scheduled flight rows; the number and percentage of missing weather-delay labels; missing departure delays, origin codes, cancellation flags, and thunderstorm flags; the number of rows flagged as missing an origin weather observation; and the number of negative weather-delay values.

Hints: `COUNT(*)` includes every row, but `COUNT(column)` excludes `NULL`. Use `100.0` when calculating percentages. A `CASE` expression can turn a condition into 1 or 0 for counting. A missing observation flag and a missing thunderstorm flag are different checks.

**Result table:** `quality`  
**Column names (in order):** `scheduled_flights`, `missing_weather_labels`, `missing_weather_label_percent`, `missing_departure_delays`, `missing_origin_codes`, `missing_cancellation_flags`, `missing_thunderstorm_flags`, `missing_origin_observations`, `negative_weather_minutes`.

In [ ]:
# Write the SQL query below. Keep these variable names and column aliases for the charts.
sql_quality = """

"""
if not sql_quality.strip():
    raise NotImplementedError("Add a SQL query to continue: Check missing and unexpected values.")
quality = con.execute(sql_quality).fetchdf()
display(quality)

## 3. How common are recorded weather delays and cancellations?

### Measure recorded weather disruptions

Return one row showing the total number of scheduled rows, the count and percentage with positive reported weather-delay minutes, and the count and percentage of weather cancellations.

A recorded weather delay means `reported_weather_delay_min > 0`. A weather cancellation requires both `cancelled = TRUE` and `cancellation_code = 'B'`. Both percentages use **all scheduled rows** as the denominator. Missing cause labels contribute no positive report, but do not prove weather had no effect.

**Result table:** `summary`  
**Column names (in order):** `scheduled_flights`, `weather_delayed_flights`, `weather_delay_percent`, `weather_cancellations`, `weather_cancellation_percent`.

In [ ]:
# Write the SQL query below. Keep these variable names and column aliases for the charts.
sql_summary = """

"""
if not sql_summary.strip():
    raise NotImplementedError("Add a SQL query to continue: Measure recorded weather disruptions.")
summary = con.execute(sql_summary).fetchdf()
display(summary)

## 4. How long are recorded weather delays?

### Describe positive weather-delay durations

Keep only positive recorded weather delays. Return their count, average, median, minimum, and maximum in minutes. Use basic aggregate functions; no quantiles or intervals are needed.

**Result table:** `severity`  
**Column names (in order):** `weather_delayed_flights`, `mean_minutes`, `median_minutes`, `minimum_minutes`, `maximum_minutes`.

In [ ]:
# Write the SQL query below. Keep these variable names and column aliases for the charts.
sql_severity = """

"""
if not sql_severity.strip():
    raise NotImplementedError("Add a SQL query to continue: Describe positive weather-delay durations.")
severity = con.execute(sql_severity).fetchdf()
display(severity)

### Count flights at each exact delay duration

For each positive value of `reported_weather_delay_min`, count the flight rows with exactly that value. Sort by duration from smallest to largest. Name the duration column `delay_minutes` and the count column `flights`. Do not create bins.

The next cell draws the bar plot for you.

**Result table:** `delay_counts`  
**Column names (in order):** `delay_minutes`, `flights`.

In [ ]:
# Write the SQL query below. Keep these variable names and column aliases for the charts.
sql_delay_counts = """

"""
if not sql_delay_counts.strip():
    raise NotImplementedError("Add a SQL query to continue: Count flights at each exact delay duration.")
delay_counts = con.execute(sql_delay_counts).fetchdf()
display(delay_counts)

**Visualise the results.** Run this cell after creating the result tables above. The chart is ready to use.

In [ ]:
plt.figure(figsize=(12, 4))
plt.bar(delay_counts['delay_minutes'], delay_counts['flights'], width=1)
plt.xlabel('Reported weather-delay minutes')
plt.ylabel('Number of flight rows')
plt.title('Number of flights at each weather-delay duration')
plt.tight_layout()
plt.show()

## 5. Which departure airports are most affected?

### Rank departure airports by total minutes

Group by origin airport, excluding missing origin codes. Calculate scheduled flight count and the sum of reported weather-delay minutes for each airport. Sort by total minutes from largest to smallest, putting unknown totals last (`NULLS LAST`). Return at most `TOP_N_AIRPORTS` rows.

End the query with `LIMIT ?`. The provided Python call supplies `[TOP_N_AIRPORTS]` for that parameter. Keep all departures in each airport group; do not filter to positive-delay rows.

**Result table:** `top_minutes`  
**Column names (in order):** `origin_iata`, `scheduled_flights`, `weather_delay_minutes`.

In [ ]:
# Write the SQL query below. Keep these variable names and column aliases for the charts.
sql_top_minutes = """

"""
if not sql_top_minutes.strip():
    raise NotImplementedError("Add a SQL query to continue: Rank departure airports by total minutes.")
top_minutes = con.execute(sql_top_minutes, [TOP_N_AIRPORTS]).fetchdf()
display(top_minutes)

### Rank departure airports by percentage

For each non-missing origin airport, calculate scheduled flight count, the count of positive recorded weather delays, and their percentage of all scheduled departures from that airport. Keep only airports with at least 10,000 departures. Sort by percentage from largest to smallest and return at most `TOP_N_AIRPORTS` rows using `LIMIT ?`.

Hint: filter individual rows with `WHERE`, but filter group sizes with `HAVING`. Do not filter out rows without a positive delay before calculating the percentage.

**Result table:** `top_rates`  
**Column names (in order):** `origin_iata`, `scheduled_flights`, `weather_delayed_flights`, `weather_delay_percent`.

In [ ]:
# Write the SQL query below. Keep these variable names and column aliases for the charts.
sql_top_rates = """

"""
if not sql_top_rates.strip():
    raise NotImplementedError("Add a SQL query to continue: Rank departure airports by percentage.")
top_rates = con.execute(sql_top_rates, [TOP_N_AIRPORTS]).fetchdf()
display(top_rates)

**Visualise the results.** Run this cell after creating the result tables above. The chart is ready to use.

In [ ]:
top_minutes.plot.barh(x='origin_iata', y='weather_delay_minutes', legend=False,
                      figsize=(10, max(4, 0.35 * len(top_minutes))))
plt.gca().invert_yaxis()
plt.xlabel('Total reported weather-delay minutes')
plt.ylabel('Origin airport')
plt.title(f'Top {len(top_minutes)} origins by recorded weather-delay burden')
plt.tight_layout()
plt.show()

top_rates.plot.barh(x='origin_iata', y='weather_delay_percent', legend=False,
                    figsize=(10, max(4, 0.35 * len(top_rates))))
plt.gca().invert_yaxis()
plt.xlabel('Recorded weather-delayed flights (% of scheduled departures)')
plt.ylabel('Origin airport')
plt.title(f'Top {len(top_rates)} reporting rates: at least 10,000 departures')
plt.tight_layout()
plt.show()

## 6. Does the reporting rate vary by month?

### Compare monthly weather-delay percentages

Use the existing `month` column. For each month, return scheduled flight count and the percentage of rows with a positive reported weather delay. Sort chronologically from month 1 to 12.

Hint: assign 1 to a positive reported weather delay and 0 otherwise with `CASE`. The average of these values is a fraction; multiply it by 100 for a percentage. Include missing delay-cause labels in the denominator through the 0 branch. No date extraction is needed.

**Result table:** `monthly`  
**Column names (in order):** `month`, `scheduled_flights`, `weather_delay_percent`.

In [ ]:
# Write the SQL query below. Keep these variable names and column aliases for the charts.
sql_monthly = """

"""
if not sql_monthly.strip():
    raise NotImplementedError("Add a SQL query to continue: Compare monthly weather-delay percentages.")
monthly = con.execute(sql_monthly).fetchdf()
display(monthly)

**Visualise the results.** Run this cell after creating the result tables above. The chart is ready to use.

In [ ]:
monthly.plot(x='month', y='weather_delay_percent', marker='o', legend=False)
plt.xticks(range(1, 13), list(calendar.month_abbr)[1:])
plt.xlabel('Month')
plt.ylabel('Recorded weather-delayed flights (% of scheduled rows)')
plt.title(f'Monthly recorded weather-delay percentage in {YEAR}')
plt.tight_layout()
plt.show()

## Conclusion


## Further investigation

- Extend the EDA with any additional insights and visualizations that you consider appropriate.
- Repeat the airport rankings for destinations using `dest_iata`. Origin and destination totals describe the same flight rows, so do not add them together.
- Change `YEAR` to 2024, rerun the notebook, and compare the two complete years.

No additional plotting code is required.

## Finish

Run this cell when you have finished your queries and plots. Rerun setup if you need to reopen the connection later.

In [ ]:
con.close()